## Calculating a Typical Meteorological Year

This notebook walks through the process of calculating a [Typical Meteorological Year](https://nsrdb.nrel.gov/data-sets/tmy), an hourly dataset used for applications in energy and building systems modeling. Because this represents average rather than extreme conditions, an AMY dataset is not suited for designing systems to meet the worst-case conditions occurring at a location.

The TMY methodology here mirrors that of the Sandia/NREL TMY3 methodology, and uses historic and projected downscaled (bias corrected?) climate data available through the Cal-Adapt: Analytics Engine catalog. 

Please note, the Analytics Engine also has an *Average Meteorological Year* functionality. The methods shown throughout this notebook will soon replace the underlying backend `climakitae` code for the AMY in order to better address our user needs, i.e., we are working to replace the AMY with the TMY methods. We provide this walkthrough to demonstrate confidence in the "AMY to TMY" conversion process for our users. 

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import climakitae as ck
import pandas as pd

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

### Step 1: Grab and process all required input data

The TMY3 method selects a "typical" month based on nine daily indices: max, min, and mean dry bulb (air) and dew point temperatures, the max and mean wind speed, global irradiance and direct irradiance (REFS here). Some of these indices are available in the Analytics Engine catalog, and some we will need to calculate ourselves.  

#### Step 1a: Indices in catalog
- Max and min air temperature
- Max and mean wind speed
- Global irradiance
- Direct irradiance

#### Step 1b: Indices that need to be calculated
- Mean air temperature
- Max, min, and mean dew point temperature

### Step 2: Calculate cumultative distribution functions

Notes on what a CDF is here: 

#### Step 2a: Calculate long-term climatology CDFs
- Calculate for each variable
- 15-20 years is the recommended minimum, use 30? 
- Leap year data is included

#### Step 2b: Calculate CDFs for each month and variable
- give proporation of values that are less than or equal to specified value of index
- Must exclude:
    - El Chichon: May 1982 until December 1984
    - Pinatubo: June 1991 to December 1994
    - Leap year days

### Step 3: Compare climatology CDF to monthly CDF for each variable

#### Step 3a: Calculate the Finkelstein-Schafer statistic 
REF HERE
Absolute difference between long-term climatology CDF and candidate CDF divided by number of days per month

#### Step 3b: Weight the F-S statistic
WS = Weight * F-S value

- Max/min air temp, max/min dry bulb temp, max/mean wind speed 1/20 each
- Mean air temp, mean dew point 2/20 each
- Global irradiance, direct irradiance 5/20 each

#### Step 3c: Select candidate months for consideration
Once weighted, select 5 candidate months that have lowest weighted sums

#### Step 3d: Rank candidate months for selection
Rank with respect to closeness of month to long-term mean and median

"Typical" month selected by choosing among 5 months with lowest WS values the month with the smallest deviation from the long-term CDF

### Step 4: Consider persistence of weather patterns

#### Step 4a: Monthly mean and median, persistence of wx patterns
- Mean daily dry bulb temp
- Global horizontal radiation
- frequency and run length above 67th percentile/below 33rd percentile

Exclude month with longest run, month with most runs, month with zero runs

### Step 5: Concatenate months together

Months concatenated together and discontinuities at month interface are smoothed for 6 hours on either side using curve-fitting techniques

### Step 6: Generate the TMY data outputs

Generally, the following is outputted using the TMY months:
Date & time (UTC)
T2m [°C] - Dry bulb (air) temperature.
RH [%] - Relative Humidity.
G(h) [W/m2] - Global horizontal irradiance.
Gb(n) [W/m2] - Direct (beam) irradiance.
Gd(h) [W/m2] - Diffuse horizontal irradiance.
IR(h) [W/m2] - Infrared radiation downwards.
WS10m [m/s] - Windspeed.
WD10m [°] - Wind direction.
SP [Pa] - Surface (air) pressure.